# Integrated Information Theory 4.0: $\Phi$ Calculation for a Canonical XOR Network

A step-by-step implementation of the IIT 4.0 "unfolding" recipe from [iit.wiki/unfolding](https://www.iit.wiki/unfolding), using **PyPhi on the  branch** (the implementation referenced by the [[...]
"
The substrate is the canonical IIT toy network: **three XOR gates, fully connected, no self-loops, in state  = (0, 0, 0)*. This system is widely used as a benchmark because its $ symmetry makes t[...]
"
The six steps of the unfolding recipe:

1. **Existence** — define a substrate model
2. **Intrinsicality** — select a candidate complex
3. **Information** — compute intrinsic information ($, $)
4. **Integration** — compute integrated information ($\varphi_s$) via the MIP
5. **Exclusion** — identify the main complex
6. **Composition** — unfold the $\Phistructure (distinctions $\varphi_d$ + relations $\varphi_r$); $\Phi = \sum \varphi_d + \sum \varphi_r$


In [7]:
import importlib.metadata
import numpy as np
import matplotlib.pyplot as plt

import pyphi
from pyphi import Direction
from pyphi.new_big_phi import (
    system_intrinsic_information,
    sia,
    phi_structure,
    maximal_complex,
)

pyphi.config.PROGRESS_BARS = False
pyphi.config.REPR_VERBOSITY = 2

print(f"PyPhi version: {importlib.metadata.version('pyphi')}")

PyPhi version: 1.2.1.dev1470+gb78d0e342


## 1. Existence — Define a Substrate Model

The [0th postulate](https://www.iit.wiki/foundations#h.5h9ssjclzgt9) demands that the units of any candidate substrate **take and make a difference** (have cause–effect power). Operationally, w[...]
"
Our substrate is three XOR gates, fully connected with **no self-loops**. Each node updates as the XOR of the *other two*:

$$A^{t+1} = B^{t} \oplus C^{t}, \qquad
B^{t+1} = A^{t} \oplus C^{t}, \qquad
C^{t+1} = A^{t} \oplus B^{t}$$

The TPM in **state-by-node** form (rows = current state, little-endian index $a + 2b + 4c$; columns = $P(\text{node}^{t+1}=1)$):

| idx | $(A,B,C)$ | $A'$ | $B'$ | $C'$ |
|-----|-----------|------|------|------|
| 0 | (0,0,0) | 0 | 0 | 0 |
| 1 | (1,0,0) | 0 | 1 | 1 |
| 2 | (0,1,0) | 1 | 0 | 1 |
| 3 | (1,1,0) | 1 | 1 | 0 |
| 4 | (0,0,1) | 1 | 1 | 0 |
| 5 | (1,0,1) | 1 | 0 | 1 |
| 6 | (0,1,1) | 0 | 1 | 1 |
| 7 | (1,1,1) | 0 | 0 | 0 |

In [8]:
# Substrate: 3 fully connected XOR gates, no self-loops
#   A(t+1) = B(t) XOR C(t)
#   B(t+1) = A(t) XOR C(t)
#   C(t+1) = A(t) XOR B(t)
#
# TPM in state-by-node form. Rows = current state in little-endian
# ordering (state (a,b,c) -> row a + 2b + 4c).
# Columns = P(node_i = ON at t+1).
tpm = np.array([
    [0, 0, 0],   # (0,0,0)
    [0, 1, 1],   # (1,0,0)
    [1, 0, 1],   # (0,1,0)
    [1, 1, 0],   # (1,1,0)
    [1, 1, 0],   # (0,0,1)
    [1, 0, 1],   # (1,0,1)
    [0, 1, 1],   # (0,1,1)
    [0, 0, 0],   # (1,1,1)
])

# Connectivity matrix (PyPhi "from-to" convention: cm[i, j] = 1 iff i -> j).
# Fully connected, no self-loops.
cm = np.array([
    [0, 1, 1],
    [1, 0, 1],
    [1, 1, 0],
])

node_labels = ("A", "B", "C")
network = pyphi.Network(tpm, cm=cm, node_labels=node_labels)
network


Network(ExplicitTPM([[[[0. 0. 0.]
   [1. 1. 0.]]

  [[1. 0. 1.]
   [0. 1. 1.]]]


 [[[0. 1. 1.]
   [1. 0. 1.]]

  [[1. 1. 0.]
   [0. 0. 0.]]]]), cm=[[0 1 1]
 [1 0 1]
 [1 1 0]])

## 2. Intrinsicality — Select a Candidate Complex

[Intrinsicality](https://www.iit.wiki/axioms-and-postulates/intrinsicality) requires that cause–effect power be exerted **within** the candidate substrate. Operationally, we choose the candidat[...]
"
PyPhi 4.0 builds **two** TPMs internally for the resulting `Subsystem`:

- a **cause TPM** $P(\text{prev state} \mid \text{current state})$ — using Bayes' rule on the conditional distribution of any background units, and
- an **effect TPM** $P(\text{next state} \mid \text{current state})$ — straightforward conditioning on the current state of any background units.

Our candidate is the entire network $\{A, B, C\}$ in current state $(0, 0, 0)$, so there are no background units — but the cause/effect distinction will still matter at every later step.

In [9]:
state = (0, 0, 0)                                 # all gates OFF
subsystem = pyphi.Subsystem(network, state, nodes=node_labels)
subsystem

Subsystem(A, B, C)

## 3. Information — Compute Intrinsic Information

The [Information postulate](https://www.iit.wiki/axioms-and-postulates/information) says: among all candidate cause and effect states, the system specifies the one that maximizes **intrinsic info[...]
"
$$ii \;=\; \underbrace{\log\frac{p_\text{constrained}}{p_\text{unconstrained}}}_{\text{informativeness}}\;\times\;\underbrace{p_\text{constrained}}_{\text{selectivity}}$$

PyPhi 4.0 computes this for the whole system via `pyphi.new_big_phi.system_intrinsic_information(subsystem)`, which returns a `SystemStateSpecification` with a maximal cause state ($ii_c$) and a [...]

In [10]:
system_state = system_intrinsic_information(subsystem)
system_state

┌────────────────────────┐
│ Specified System State │
│   ╍╍╍╍╍╍╍╍╍╍╍╍╍╍╍╍╍    │
│    CAUSE:  (0,0,0)     │
│     II_c: 1.0          │
│   EFFECT:  (0,0,0)     │
│     II_e: 2.0          │
└────────────────────────┘

## 4. Integration — Compute Integrated Information $\varphi_s$

[Integration](https://www.iit.wiki/axioms-and-postulates/integration) demands the system be **irreducible** to the cause–effect power of its parts. Operationally, partition the system along eve[...]
"
$$\varphi_s \;=\; \min_{P}\,\Big[\,\mathrm{ii}(M)\;-\;\mathrm{ii}(M / P)\,\Big]$$

In PyPhi 4.0 this is `subsystem.sia()` (also available as `pyphi.new_big_phi.sia(subsystem)`). The returned `SystemIrreducibilityAnalysis` object holds:

- `.phi` — $\varphi_s$
- `.normalized_phi` — $\varphi_s$ divided by the max possible for a system of this size
- `.partition` — the MIP itself (a `SystemPartition` showing which connections are severed)
- `.cause`, `.effect` — per-direction `RepertoireIrreducibilityAnalysis` objects
- `.system_state` — the cause/effect state from step 3
- `.ties` — list of all MIPs tied for the minimum (often non-trivial in symmetric networks like ours)

In [11]:
sia_result = subsystem.sia()
sia_result

┌───────────────────────────────────┐
│ SystemIrreducibilityAnalysis      │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━        │
│       Subsystem:  A,B,C           │
│   Current state:  (0,0,0)         │
│             φ_s: 1.5              │
│  Normalized φ_s: 0.25             │
│           CAUSE:  (0,0,0)         │
│            II_c: 1.0              │
│          EFFECT:  (0,0,0)         │
│            II_e: 2.0              │
│    #(tied MIPs): 0                │
│       Partition:                  │
│                  3 parts: {A,B,C} │
│                  [[0 1 1]         │
│                   [1 0 1]         │
│                   [1 1 0]]        │
└───────────────────────────────────┘

## 5. Exclusion — Identify the Main Complex

[Exclusion](https://www.iit.wiki/axioms-and-postulates/exclusion) requires that the substrate of consciousness be **definite**: of all overlapping candidate complexes (subsets, supersets, paraset[...]
"
Operationally, this is a search across all subsystems of the network. PyPhi 4.0 provides `pyphi.new_big_phi.maximal_complex(network, state)`, which not only finds the main complex but **also retu[...]

In [19]:
main_sia = maximal_complex(network, state)

main_nodes = tuple(node_labels[i] for i in main_sia.node_indices)
print(f"Main complex nodes : {main_nodes}")
print(f"phi_s              : {float(main_sia.phi):.6f}")
print(f"Normalized phi_s   : {float(main_sia.normalized_phi):.6f}")
print(f"# tied MIPs        : {len(main_sia.ties)}")

assert main_sia.node_indices == (0, 1, 2), \
    "Expected the whole network to be the main complex for 3 fully-connected XORs."

Main complex nodes : ('A', 'B', 'C')
phi_s              : 1.500000
Normalized phi_s   : 0.250000
# tied MIPs        : 1


## 6. Composition — Unfold the $\Phi$-structure

[Composition](https://www.iit.wiki/axioms-and-postulates/composition) requires that the cause–effect power of a complex be **structured** — composed of:

- **Distinctions** ($\varphi_d$): each irreducible mechanism (subset of units in the mechanism powerset) specifies a maximally-irreducible cause purview and effect purview, congruent with the sy[...]
"
- **Relations** ($\varphi_r$): congruent overlaps between distinction purviews (2nd-degree, 3rd-degree faces, etc.) — measuring how much joint cause-effect power the distinctions exert togethe[...]
"
The full $\Phi$-structure is then the union of all irreducible distinctions and relations:

$$\Phi \;=\; \sum_{d}\varphi_d \;+\; \sum_{r}\varphi_r$$

This is the *qualitative* analog of consciousness — its specific structure, not just the quantity. We compute it via `pyphi.new_big_phi.phi_structure(subsystem)` on the main complex's subsystem[...]

In [20]:
# Build the subsystem of the main complex (same as `subsystem` here, since the
# main complex is the whole network) and unfold its full Phi-structure.
# Passing the already-computed SIA avoids re-running the integration step.
main_subsystem = pyphi.Subsystem(network, state, nodes=main_sia.node_indices)
ps = phi_structure(main_subsystem, sia=main_sia)

print(f"Big Phi = Σφ_d + Σφ_r = {float(ps.big_phi):.6f}")
print(f"  Σφ_d : {float(ps.sum_phi_distinctions):.6f}")
print(f"  Σφ_r : {float(ps.sum_phi_relations):.6f}")
print()

def fmt_nodes(items):
    # Distinction purviews yield int indices; relation purviews yield Unit objects (sets).
    def to_idx(x):
        return x.index if hasattr(x, "index") else int(x)
    indices = sorted(to_idx(i) for i in items)
    return "{" + ",".join(node_labels[i] for i in indices) + "}"

print(f"=== {len(ps.distinctions)} distinctions ===\n")
print(f"  {'mechanism':<10} {'cause':<10} {'effect':<10} {'phi_d':>10}")
print("  " + "-" * 44)

dist_rows = []
for d in sorted(ps.distinctions, key=lambda c: -c.phi):
    mech     = fmt_nodes(d.mechanism)
    cause_p  = fmt_nodes(d.cause.purview)
    effect_p = fmt_nodes(d.effect.purview)
    dist_rows.append((mech, float(d.phi)))
    print(f"  {mech:<10} {cause_p:<10} {effect_p:<10} {float(d.phi):>10.6f}")

Big Phi = Σφ_d + Σφ_r = 9.500000
  Σφ_d : 2.500000
  Σφ_r : 7.000000

=== 4 distinctions ===

  mechanism  cause      effect          phi_d
  --------------------------------------------
  {A,B,C}    {A,B,C}    {A,B,C}      1.000000
  {A,B}      {A,B,C}    {C}          0.500000
  {A,C}      {A,B,C}    {B}          0.500000
  {B,C}      {A,B,C}    {A}          0.500000


In [21]:
n_rel = ps.relations.num_relations()
print(f"=== {n_rel} relations (sum φ_r = {float(ps.sum_phi_relations):.6f}) ===\n")
print(f"  {'degree':>6}  {'purview':<10}  {'#faces':>7}  {'phi_r':>10}")
print("  " + "-" * 42)

rel_rows = []
for r in sorted(ps.relations, key=lambda r: -float(r.phi)):
    purview = fmt_nodes(r.purview)
    degree  = len(r)                # Relation is itself a frozenset of distinctions
    n_faces = r.num_faces
    rel_rows.append((degree, purview, float(r.phi)))
    print(f"  {degree:>6}  {purview:<10}  {n_faces:>7}  {float(r.phi):>10.6f}")

=== 15 relations ===

  degree  purview          phi_r
  --------------------------------


AttributeError: 'Relation' object has no attribute 'relata'

In [ ]:
# Visualize the Phi-structure: sum of distinction phi_d and relation phi_r = Big Phi
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

mechs = [m for m, _ in dist_rows]
phis_d = [p for _, p in dist_rows]
axes[0].bar(mechs, phis_d, color="steelblue", edgecolor="black")
axes[0].set_ylabel(r"$\varphi_d$")
axes[0].set_xlabel("Mechanism")
axes[0].set_title(rf"Distinctions ($\Sigma\varphi_d = {sum(phis_d):.3f}$)")

if rel_rows:
    deg_to_phi = {}
    for deg, _, p in rel_rows:
        deg_to_phi.setdefault(deg, []).append(p)
    degrees = sorted(deg_to_phi)
    sums = [sum(deg_to_phi[d]) for d in degrees]
    axes[1].bar([str(d) for d in degrees], sums, color="indianred", edgecolor="black")
    axes[1].set_xlabel("Relation degree")
    axes[1].set_ylabel(r"$\Sigma\varphi_r$")
    axes[1].set_title(rf"Relations ($\Sigma\varphi_r = {sum(sums):.3f}$)")
else:
    axes[1].text(0.5, 0.5, "no relations", ha="center", va="center")
    axes[1].set_axis_off()

big_phi = float(ps.big_phi)
axes[2].bar(["Σφ_d", "Σφ_r", "Φ"],
            [float(ps.sum_phi_distinctions), float(ps.sum_phi_relations), big_phi],
            color=["steelblue", "indianred", "darkorange"], edgecolor="black")
axes[2].set_title(rf"Total $\Phi = {big_phi:.3f}$")
axes[2].set_ylabel("integrated information (bits)")

plt.tight_layout()
plt.show()

## Summary

Reproducing the canonical IIT 4.0 analysis of three fully-connected XOR gates, no self-loops, state  = (0, 0, 0)$:

- **System integrated information** $\varphi_s$ (Step 4): the system is irreducible ($\varphi_s > 0$), qualifying it as a complex.
- **Main complex** (Step 5): the whole $\{A, B, C\}$ — exclusion retains all units, consistent with the network's $ symmetry.
- **$\Phistructure** (Step 6):
  - **Distinctions**: every congruent subset of $\{A, B, C\}$ yields an irreducible distinction ($\varphi_d > 0$) — a direct consequence of XOR's non-decomposability.
  - **Relations**: 2nd- and 3rd-degree relations between overlapping distinction purviews — the structural "glue" first captured in IIT 4.0.
- **Big $\Phi* $= \sum \varphi_d + \sum \varphi_r$ — the total integrated information of the system's cause-effect structure.

### Key differences from IIT 3.0

| | IIT 3.0 (PyPhi 1.2.0) | IIT 4.0 (PyPhi ) |
|---|---|---|
| Distance measure | Earth-mover's distance | Intrinsic difference (default) |
| System irreducibility | Bipartitions only | Directed *k*-cuts (richer partition space) |
| Relations | Not computed | First-class objects with $\varphi_r$ |
| $\Phi$ | $\varphi_s$ across the MIP | $\sum \varphi_d + \sum \varphi_r$ |

Reference: [Albantakis et al., *PLoS Comp Bio* 19(10): e1011465 (2023)](https://doi.org/10.1371/journal.pcbi.1011465).

---

*This implementation is part of a broader investigation into information-theoretic measures of emergence and integration in both biological and artificial neural systems. The XOR network serves a[...]
